In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

path = '../data/05_processed/funil_por_regiao.parquet'

df = pd.read_parquet(path)

display(df.columns.to_list())#type: ignore
display(df.head())#type: ignore

# --- 1. Agregar somando os valores base (para cada regiao_ies, área e período) ---
colunas_base = [
    'vagas_fies',
    'Inscritos_Geral',
    'inscritos_com_nota_suficiente',
    'Candidatos_Unicos_Geral',
    'candidatos_unicos_com_nota_suficiente',
    'vagas_ocupadas'
]

df_agg = (
    df.groupby(['ano', 'semestre', 'regiao_ies', 'nome_cine_area_geral'], as_index=False)[colunas_base]
    .sum()
)

# Substitui possíveis resultados 'inf' por NaN (nulo)
df_agg = df_agg.replace([np.inf, -np.inf], np.nan)

# --- 2. Calcular taxas CORRETAMENTE a partir dos somatórios ---
df_agg['taxa_inscricao'] = df_agg['Inscritos_Geral'] / df_agg['vagas_fies']
df_agg['taxa_aprovacao_por_inscritos'] = df_agg['inscritos_com_nota_suficiente'] / df_agg['Inscritos_Geral']
df_agg['taxa_aprovacao_por_candidato'] = df_agg['candidatos_unicos_com_nota_suficiente'] / df_agg['Candidatos_Unicos_Geral']
df_agg['taxa_ocupacao'] = df_agg['vagas_ocupadas'] / df_agg['vagas_fies']
df_agg['taxa_conversao_inscritos'] = df_agg['vagas_ocupadas'] / df_agg['Inscritos_Geral']
df_agg['taxa_conversao_candidatos'] = df_agg['vagas_ocupadas'] / df_agg['Candidatos_Unicos_Geral']
df_agg['taxa_inscritos_capacitados'] = df_agg['vagas_ocupadas'] / df_agg['inscritos_com_nota_suficiente']
df_agg['taxa_candidatos_capacitados'] = df_agg['vagas_ocupadas'] / df_agg['candidatos_unicos_com_nota_suficiente']


display(df_agg)#type: ignore

# -------------------------------------------------------------------------
# 1. PREPARAÇÃO DOS DADOS (Criação do df_completo, período e taxas)
# -------------------------------------------------------------------------
df_plot = df_agg.copy()

# --- Criar coluna de período (ex: 2019.1) ---
df_plot['periodo'] = df_plot['ano'].astype(str) + '.' + df_plot['semestre'].astype(str)

# --- Linha NACIONAL: média de todas as regiões para cada área CINE ---
df_nacional = (
    df_plot.groupby(['ano', 'semestre', 'nome_cine_area_geral'], as_index=False)
    .mean(numeric_only=True)
)
# Alterado o nome conforme seu pedido
df_nacional['regiao_ies'] = 'Território Nacional'
df_nacional['periodo'] = df_nacional['ano'].astype(str) + '.' + df_nacional['semestre'].astype(str)

# --- Linha TOTAL: média de todas as áreas e regiões (única linha global) ---
df_total = (
    df_plot.groupby(['ano', 'semestre'], as_index=False)
    .mean(numeric_only=True)
)
# Alterado o nome conforme seu pedido
df_total['regiao_ies'] = 'Média de Todos os Cursos (Nacional)'
df_total['nome_cine_area_geral'] = 'Todas as Áreas do CINE'
df_total['periodo'] = df_total['ano'].astype(str) + '.' + df_total['semestre'].astype(str)

# --- Unir tudo ---
df_completo = pd.concat([df_plot, df_nacional, df_total], ignore_index=True)

# --- Lista das taxas ---
taxas = [
    'taxa_inscricao',
    'taxa_aprovacao_por_inscritos',
    'taxa_aprovacao_por_candidato',
    'taxa_ocupacao',
    'taxa_conversao_inscritos',
    'taxa_conversao_candidatos',
    'taxa_inscritos_capacitados',
    'taxa_candidatos_capacitados'
]

# --- Converter para porcentagem ---
df_completo[taxas] = df_completo[taxas] * 100

# -------------------------------------------------------------------------
# 2. CONFIGURAÇÕES VISUAIS (Nível Artigo Acadêmico)
# -------------------------------------------------------------------------

# --- Pasta base para os gráficos ---
base_dir = "../reports/figures/analise_021"
os.makedirs(base_dir, exist_ok=True)

# --- Cores por região (Atualizado com os novos nomes) ---
cores = {
    'Norte': '#1f77b4',
    'Nordeste': '#ff7f0e',
    'Centro-Oeste': '#2ca02c',
    'Sudeste': '#d62728',
    'Sul': '#9467bd',
    'Território Nacional': '#000000',                   # Preto
    'Média de Todos os Cursos (Nacional)': '#808080'    # Cinza
}

# --- Estilos de linha (Atualizado com os novos nomes) ---
dashes = {
    'Norte': '',
    'Nordeste': '',
    'Centro-Oeste': '',
    'Sudeste': '',
    'Sul': '',
    'Território Nacional': (3, 2),                   # Pontilhado curto
    'Média de Todos os Cursos (Nacional)': (5, 3)    # Tracejado longo
}

# Dicionário para limpar o nome das taxas no título
nomes_taxas_limpos = {
    'taxa_inscricao': 'Taxa de Inscrição',
    'taxa_aprovacao_por_inscritos': 'Taxa de Aprovação por Inscritos',
    'taxa_aprovacao_por_candidato': 'Taxa de Aprovação por Candidato',
    'taxa_ocupacao': 'Taxa de Ocupação',
    'taxa_conversao_inscritos': 'Conversão de Inscritos',
    'taxa_conversao_candidatos': 'Conversão de Candidatos',
    'taxa_inscritos_capacitados': 'Inscritos Capacitados',
    'taxa_candidatos_capacitados': 'Candidatos Capacitados'
}

# --- Estilo: Fundo branco limpo, sem grades pesadas ---
sns.set_theme(style="ticks", palette="deep")

# -------------------------------------------------------------------------
# 3. GERAÇÃO DOS GRÁFICOS
# -------------------------------------------------------------------------

for taxa in taxas:
    taxa_dir = os.path.join(base_dir, taxa)
    os.makedirs(taxa_dir, exist_ok=True)
    
    # Pega o nome limpo para o título
    taxa_nome = nomes_taxas_limpos.get(taxa, taxa)

    for area in df_completo['nome_cine_area_geral'].unique():
        df_area = df_completo[
            df_completo['nome_cine_area_geral'].isin([area, 'Todas as Áreas do CINE'])
        ]

        # 1. Aumentei a altura para 7 (para dar mais espaço vertical) e a largura para 11
        fig, ax = plt.subplots(figsize=(11, 7))

        # Plot principal
        g = sns.lineplot(
            data=df_area,
            x='periodo',
            y=taxa,
            hue='regiao_ies',
            style='regiao_ies',
            dashes=dashes,
            palette=cores,
            markers=True,
            linewidth=2.0,
            ax=ax
        )

        # Aumentar espessura das linhas “Nacionais”
        for line, label in zip(g.lines, g.get_legend_handles_labels()[1]):
            if label in ['Território Nacional', 'Média de Todos os Cursos (Nacional)']:
                line.set_linewidth(3.5)

        # --- Ajustes de Legenda ---
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles=handles, 
            labels=labels,
            title=None,                     
            loc='lower center', 
            bbox_to_anchor=(0.5, 1.02), # Legenda fica logo acima da linha do gráfico    
            ncol=3,                         
            frameon=False                   
        )

        # --- Título e Eixos Clean ---
        # 2. Aumentei o 'pad' para 85. Isso empurra o título bem para cima, pulando a legenda.
        plt.title(f"{taxa_nome}\n{area}", fontsize=14, weight='bold', loc='center', pad=85)
        
        plt.xlabel('Semestre Letivo', fontsize=11)
        plt.ylabel('Taxa (%)', fontsize=11)
        
        # Eixo x na horizontal
        plt.xticks(rotation=0)
        
        # Remove a linha superior e direita
        sns.despine() 

        # 3. Ajuste do layout antes de salvar
        plt.tight_layout()

        # Caminho do arquivo
        file_name = f"grafico_{str(area).replace('/', '_').replace(' ', '_')}.png"
        file_path = os.path.join(taxa_dir, file_name)
        
        # 4. DPI aumentado para 400 (Nível altíssimo para publicações)
        plt.savefig(file_path, dpi=400, bbox_inches='tight')
        plt.close()

print("✅ Gráficos corrigidos e salvos em alta resolução!")

['ano',
 'semestre',
 'nome_cine_area_geral',
 'regiao_ies',
 'Inscritos_Geral',
 'inscritos_com_nota_suficiente',
 'Candidatos_Unicos_Geral',
 'candidatos_unicos_com_nota_suficiente',
 'inscritos_gap_menos_100',
 'vol_inscritos_contratada',
 'vol_inscritos_inscricao_postergada',
 'vol_inscritos_lista_de_espera',
 'vol_inscritos_nao_contratado',
 'vol_inscritos_opcao_nao_contratada',
 'vol_inscritos_participacao_cancelada_pelo_candidato',
 'vol_inscritos_pre_selecionado',
 'vol_inscritos_rejeitada_pela_cpsa',
 'vagas_fies',
 'vagas_ocupadas']

,ano,semestre,nome_cine_area_geral,regiao_ies,Inscritos_Geral,inscritos_com_nota_suficiente,Candidatos_Unicos_Geral,candidatos_unicos_com_nota_suficiente,inscritos_gap_menos_100,vol_inscritos_contratada,vol_inscritos_inscricao_postergada,vol_inscritos_lista_de_espera,vol_inscritos_nao_contratado,vol_inscritos_opcao_nao_contratada,vol_inscritos_participacao_cancelada_pelo_candidato,vol_inscritos_pre_selecionado,vol_inscritos_rejeitada_pela_cpsa,vagas_fies,vagas_ocupadas
0,2019,1,"Agricultura, silvicultura, pesca e veterinária",Centro-Oeste,2575,504,1627,378,704.0,211,0,1347,888,60,61,3,5,1973.0,214.0
1,2019,1,"Agricultura, silvicultura, pesca e veterinária",Nordeste,4485,1777,2695,1102,672.0,660,0,1558,1710,408,143,0,6,3559.0,660.0
2,2019,1,"Agricultura, silvicultura, pesca e veterinária",Norte,2219,582,1292,340,247.0,224,0,885,921,147,37,0,5,951.0,225.0
3,2019,1,"Agricultura, silvicultura, pesca e veterinária",Sudeste,11070,1817,6409,1242,3701.0,691,0,7327,2473,316,205,1,57,5398.0,696.0
4,2019,1,"Agricultura, silvicultura, pesca e veterinária",Sul,3333,607,2199,444,644.0,331,0,1739,1028,124,103,0,8,2697.0,332.0


,ano,semestre,regiao_ies,nome_cine_area_geral,vagas_fies,Inscritos_Geral,inscritos_com_nota_suficiente,Candidatos_Unicos_Geral,candidatos_unicos_com_nota_suficiente,vagas_ocupadas,taxa_inscricao,taxa_aprovacao_por_inscritos,taxa_aprovacao_por_candidato,taxa_ocupacao,taxa_conversao_inscritos,taxa_conversao_candidatos,taxa_inscritos_capacitados,taxa_candidatos_capacitados
0,2019,1,Centro-Oeste,"Agricultura, silvicultura, pesca e veterinária",1973.0,2575,504,1627,378,214.0,1.305119,0.195728,0.232329,0.108464,0.083107,0.131530,0.424603,0.566138
1,2019,1,Centro-Oeste,Artes e humanidades,1664.0,404,212,213,122,27.0,0.242788,0.524752,0.572770,0.016226,0.066832,0.126761,0.127358,0.221311
2,2019,1,Centro-Oeste,"Ciências naturais, matemática e estatística",560.0,232,53,105,25,7.0,0.414286,0.228448,0.238095,0.012500,0.030172,0.066667,0.132075,0.280000
3,2019,1,Centro-Oeste,"Ciências sociais, comunicação e informação",3943.0,3853,569,1957,361,178.0,0.977175,0.147677,0.184466,0.045143,0.046198,0.090956,0.312830,0.493075
4,2019,1,Centro-Oeste,Computação e Tecnologias da Informação e Comun...,3945.0,1770,685,803,346,78.0,0.448669,0.387006,0.430884,0.019772,0.044068,0.097136,0.113869,0.225434
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,2021,2,Sul,Educação,3122.0,166,117,113,83,18.0,0.053171,0.704819,0.734513,0.005766,0.108434,0.159292,0.153846,0.216867
296,2021,2,Sul,"Engenharia, produção e construção",11069.0,1243,817,699,478,188.0,0.112296,0.657281,0.683834,0.016984,0.151247,0.268956,0.230110,0.393305
297,2021,2,Sul,"Negócios, administração e direito",23468.0,3800,1322,1878,687,545.0,0.161923,0.347895,0.365815,0.023223,0.143421,0.290202,0.412254,0.793304
298,2021,2,Sul,Saúde e bem-estar,12383.0,10510,3361,6511,1903,1007.0,0.848744,0.319791,0.292275,0.081321,0.095814,0.154661,0.299613,0.529164


✅ Gráficos corrigidos e salvos em alta resolução!
